In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML


dt=1e-4 ; m=1.0 ; ell=2.0 ; alpha=np.pi/4 ; g=9.81 ;      # m1 = m2 = m ; ℓ = longitud de la cuerda ; α = semiángulo del cono
tf=8; N=int(tf/dt);

t=np.linspace(0,tf,N)


# Condiciones iniciales  (r1, θ1, φ1, φ2, ṙ1, θ̇1, φ̇1, φ̇2)      θ1 se mide desde +z  ->  m1 colgando: θ1 ≈ π
r10      = 1.0                                              # r1(0): m1 a 1 m de O  (m2 a ℓ − r1 = 1 m de O)
om1      = np.sqrt(g/r10)                                   # frecuencia de las oscilaciones de m1 (apartado e)
phd2_cir = np.sqrt(g/((ell - r10)*(1 + np.cos(alpha))))     # φ̇2 para que m2 describa un círculo horizontal (apartado e)

def caso_e(delta0):
    """Apartado (e): m1 se suelta con θ1 = π − δ0, φ̇1(0) = 0 ; m2 en trayectoria circular."""
    return dict(label=f'(e)  $\\delta_0={delta0}$', r1=r10, th1=np.pi - delta0, ph1=0.0, ph2=0.0,
                rd1=0.0, thd1=0.0, phd1=0.0, phd2=phd2_cir)

casos = [caso_e(0.1), caso_e(0.025), caso_e(0.05), caso_e(0.2),
         dict(label='caso general', r1=1.0, th1=np.pi - 0.3, ph1=0.0, ph2=0.0, rd1=0.3, thd1=0.0, phd1=1.5, phd2=1.8),
         # --- situaciones iniciales para las animaciones ---
         {**caso_e(0.15), 'label': 'animación: pequeñas oscilaciones'},
         dict(label='animación: movimiento cualquiera', r1=1.0, th1=np.pi - 0.25, ph1=0.0, ph2=0.0,
              rd1=0.2, thd1=0.5, phd1=1.2, phd2=1.5)]
i_main, i_gen = 0, 4                                        # caso principal (e) y caso general (gráficas)
i_anim_osc, i_anim_gen = 5, 6                               # casos que se animan: oscilaciones pequeñas / movimiento cualquiera

## FUNCIONES GENERALES

In [ ]:
def lagr_hamil(T, U, qs, qds, ps):
    """
    A partir de T y U devuelve (L, p, H, sol):
      L   = T − U
      p   = [∂L/∂q̇_i]                      momentos generalizados (en función de q, q̇)
      H   = Σ p_i q̇_i − L  expresado en función de (q, p)   <- siempre en momentos generalizados
      sol = {q̇_i: q̇_i(q, p)}               inversión de p = p(q, q̇)
    qs, qds, ps: listas de símbolos de sympy (coordenadas, velocidades y momentos canónicos).
    """
    L = sp.simplify(T - U)
    p = [sp.diff(L, qd) for qd in qds]
    sol = sp.solve([sp.Eq(P, pe) for P, pe in zip(ps, p)], qds, dict=True)[0]
    H_vel = sum(pe*qd for pe, qd in zip(p, qds)) - L                    # H(q, q̇)
    H = sp.simplify(H_vel.subs(sol))                                    # H(q, p)
    return L, p, H, sol


def euler_lagrange(L, qs, qds, qdds):
    """
    Ecuaciones de Euler-Lagrange   d/dt(∂L/∂q̇_i) − ∂L/∂q_i = 0.
    Devuelve la lista de expresiones EL_i (cada una igualada a 0), con q̈_i = qdds[i].
    La derivada temporal total se hace con la regla de la cadena sobre (q, q̇).
    """
    n = len(qs)
    p = [sp.diff(L, qd) for qd in qds]
    EL = []
    for i in range(n):
        ddt_p = sum(sp.diff(p[i], qs[j])*qds[j] + sp.diff(p[i], qds[j])*qdds[j] for j in range(n))
        EL.append(sp.simplify(ddt_p - sp.diff(L, qs[i])))
    return EL


def hamilton_eqs(H, qs, ps):
    """Ecuaciones de Hamilton  q̇_i = ∂H/∂p_i ,  ṗ_i = −∂H/∂q_i.  Devuelve (qdot, pdot)."""
    qdot = [sp.diff(H, P) for P in ps]
    pdot = [-sp.diff(H, q) for q in qs]
    return qdot, pdot


def poisson(f, g, qs, ps):
    """Paréntesis de Poisson  {f, g} = Σ_i (∂f/∂q_i ∂g/∂p_i − ∂f/∂p_i ∂g/∂q_i)."""
    return sp.simplify(sum(sp.diff(f, q)*sp.diff(g, P) - sp.diff(f, P)*sp.diff(g, q) for q, P in zip(qs, ps)))

## LAGRANGIANO, HAMILTONIANO, ECUACIONES DE EULER-LAGRANGE Y DE HAMILTON

In [ ]:
#(b), (c) Lagrangiano, momentos y Hamiltoniano   (H = Σ p_i q̇_i − L)
r1_s, th1_s, ph1_s, ph2_s = sp.symbols('r1 theta1 phi1 phi2', real=True)
rd1_s, thd1_s, phd1_s, phd2_s = sp.symbols('r1dot theta1dot phi1dot phi2dot', real=True)
rdd1_s, thdd1_s, phdd1_s, phdd2_s = sp.symbols('r1ddot theta1ddot phi1ddot phi2ddot', real=True)
pr_s, pth_s, pph1_s, pph2_s = sp.symbols('p_r p_theta p_phi1 p_phi2', real=True)
m_, l_, al_, g_ = sp.symbols('m ell alpha g', positive=True)

def v2(r, rd, th, thd, phd):                          # |v|² en esféricas
    return rd**2 + r**2*thd**2 + r**2*sp.sin(th)**2*phd**2

# Ligaduras:  cuerda  r1 + r2 = ℓ   y   cono  θ2 = π − α   (θ̇2 = 0)   ->   4 grados de libertad: (r1, θ1, φ1, φ2)
r2_s = l_ - r1_s
th2  = sp.pi - al_

T_sym = sp.simplify(sp.Rational(1,2)*m_*v2(r1_s, rd1_s, th1_s, thd1_s, phd1_s)
                  + sp.Rational(1,2)*m_*v2(r2_s, -rd1_s, th2, 0, phd2_s))
U_sym = sp.simplify(m_*g_*(r1_s*sp.cos(th1_s) + r2_s*sp.cos(th2)))
E_sym = T_sym + U_sym
print('T =', T_sym)
print('U =', U_sym)

qs, qds, qdds, ps = [r1_s, th1_s, ph1_s, ph2_s], [rd1_s, thd1_s, phd1_s, phd2_s], [rdd1_s, thdd1_s, phdd1_s, phdd2_s], [pr_s, pth_s, pph1_s, pph2_s]

# L, p, H(q,p) con la función general
L_sym, p_expr, H_can, sol = lagr_hamil(T_sym, U_sym, qs, qds, ps)
H_vel = sp.simplify(sum(p*qd for p, qd in zip(p_expr, qds)) - L_sym)      # H(q, q̇)
print('p =', p_expr)
print('H − E =', sp.simplify(H_vel - E_sym), '   ->  H = E' if sp.simplify(H_vel - E_sym) == 0 else '   ->  H ≠ E')

# (d) Ecuaciones de Euler-Lagrange:   d/dt(∂L/∂q̇_i) − ∂L/∂q_i = 0
EL = euler_lagrange(L_sym, qs, qds, qdds)

eq_th1 = r1_s*thdd1_s + 2*rd1_s*thd1_s - r1_s*phd1_s**2*sp.sin(th1_s)*sp.cos(th1_s) - g_*sp.sin(th1_s)
rest_r = 2*rdd1_s - r1_s*(thd1_s**2 + phd1_s**2*sp.sin(th1_s)**2) + g_*(sp.cos(th1_s) + sp.cos(al_))
h_sym  = sp.simplify((EL[0]/m_ - rest_r)/(phd2_s**2*sp.sin(al_)**2))      # el resto de la ecuación en r1 es h(r1) φ̇2² sin²α

print('EL_θ1 / (m r1) − [r1 θ̈1 + 2 ṙ1 θ̇1 − r1 φ̇1² sinθ1 cosθ1 − g sinθ1] =', sp.simplify(EL[1]/(m_*r1_s) - eq_th1))
print('h(r1) =', h_sym)
print('∂L/∂φ1 =', sp.diff(L_sym, ph1_s), ' ,  ∂L/∂φ2 =', sp.diff(L_sym, ph2_s), '   ->  φ1 y φ2 cíclicas:  p_φ1 y p_φ2 se conservan')

# (c) Hamiltoniano en variables canónicas:  q̇ = q̇(q, p)
H_nice = (pr_s**2/(4*m_) + pth_s**2/(2*m_*r1_s**2) + pph1_s**2/(2*m_*r1_s**2*sp.sin(th1_s)**2)
          + pph2_s**2/(2*m_*(l_ - r1_s)**2*sp.sin(al_)**2) + m_*g_*(r1_s*(sp.cos(th1_s) + sp.cos(al_)) - l_*sp.cos(al_)))
print('H(q, p) = p_r²/(4m) + p_θ²/(2 m r1²) + p_φ1²/(2 m r1² sin²θ1) + p_φ2²/(2 m (ℓ−r1)² sin²α) + m g [r1 (cosθ1 + cosα) − ℓ cosα]')
print('(comprobación simbólica)  H(q, p) − expresión anterior =', sp.simplify(H_can - H_nice))

# Ecuaciones de Hamilton:  q̇ = ∂H/∂p ,  ṗ = −∂H/∂q
qdot_eq, pdot_eq = hamilton_eqs(H_can, qs, ps)

par_s = (m_, l_, al_, g_)
f_num  = sp.lambdify((*qs, *ps, *par_s), qdot_eq + pdot_eq, 'numpy', cse=True)    # (ṙ1, θ̇1, φ̇1, φ̇2, ṗ_r, ṗ_θ, ṗ_φ1, ṗ_φ2)
H_num  = sp.lambdify((*qs, *ps, *par_s), H_can, 'numpy')
qd_num = sp.lambdify((*qs, *ps, *par_s), [sol[qd] for qd in qds], 'numpy')
E_num  = sp.lambdify((*qs, *qds, *par_s), E_sym, 'numpy')
L_num  = sp.lambdify((*qs, *qds, *par_s), L_sym, 'numpy')

## PARÉNTESIS DE POISSON

In [ ]:
# Paréntesis fundamentales {q_i, p_j} = δ_ij
PB = sp.Matrix(4, 4, lambda i, j: poisson(qs[i], ps[j], qs, ps))
print('{q_i, p_j} =', PB.tolist())

# Constantes del movimiento:  df/dt = {f, H}  (si f no depende explícitamente de t)
for nombre, f in (('p_φ1', pph1_s), ('p_φ2', pph2_s), ('p_r', pr_s), ('p_θ', pth_s)):
    pb = poisson(f, H_can, qs, ps)
    print(f'{{{nombre}, H}} =', pb if pb == 0 else '≠ 0   (no se conserva; es ṗ)')

# Las ecuaciones de Hamilton en forma de Poisson:  q̇ = {q, H} ,  ṗ = {p, H}
ok = all(sp.simplify(poisson(q, H_can, qs, ps) - qd) == 0 for q, qd in zip(qs, qdot_eq)) and \
     all(sp.simplify(poisson(P, H_can, qs, ps) - pd) == 0 for P, pd in zip(ps, pdot_eq))
print('q̇ = {q, H} y ṗ = {p, H} coinciden con las ecuaciones de Hamilton:', ok)
print('{p_φ1, p_φ2} =', poisson(pph1_s, pph2_s, qs, ps), ' (las dos cantidades conservadas están en involución)')

## FUNCION PARA EL MOVIMIENTO (RUNGE-KUTTA 4)

In [ ]:
def F(y, pars):
    """Segundo miembro de las ecuaciones de Hamilton. y: (8, ncasos) = (r1, θ1, φ1, φ2, p_r, p_θ, p_φ1, p_φ2)."""
    d = f_num(*y, *pars)
    return np.array([np.broadcast_to(v, y.shape[1:]) for v in d])


def Motion(y0, pars, t):
    """Integra las ecuaciones de Hamilton con Runge-Kutta de orden 4 (varios casos a la vez)."""
    nv, nc = y0.shape
    N = len(t)
    y = np.zeros((nv, nc, N))
    y[:, :, 0] = y0

    for i in range(N - 1):
        dt = t[i+1] - t[i]
        yi = y[:, :, i]

        k1 = F(yi, pars)
        k2 = F(yi + 0.5*dt*k1, pars)
        k3 = F(yi + 0.5*dt*k2, pars)
        k4 = F(yi + dt*k3, pars)

        y[:, :, i+1] = yi + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

    return y


#Estado inicial: momentos  p = ∂L/∂q̇
ci = {k: np.array([cs[k] for cs in casos]) for k in casos[0] if k != 'label'}
y0 = np.array([ci['r1'], ci['th1'], ci['ph1'], ci['ph2'],
               2*m*ci['rd1'],                                         # p_r  = 2 m ṙ1
               m*ci['r1']**2*ci['thd1'],                               # p_θ  = m r1² θ̇1
               m*ci['r1']**2*np.sin(ci['th1'])**2*ci['phd1'],           # p_φ1 = m r1² sin²θ1 φ̇1
               m*(ell - ci['r1'])**2*np.sin(alpha)**2*ci['phd2']])     # p_φ2 = m (ℓ−r1)² sin²α φ̇2

#Simulación
pars = (m, ell, alpha, g)
y = Motion(y0, pars, t)
r1, th1, ph1, ph2, pr, pth, pph1, pph2 = y

## CANTIDADES CONSERVADAS

In [ ]:
#(c) Energía, Hamiltoniano y momentos generalizados   (caso general)
qd = qd_num(*y, *pars)                                                    # velocidades generalizadas a partir de los momentos
E  = E_num(*y[:4], *qd, *pars)                                            # E = T + U
H  = H_num(*y, *pars)                                                     # H(q, p)
L  = L_num(*y[:4], *qd, *pars)

print(f'max |H − E| / |E0|  (todos los casos) = {np.abs(H - E).max()/np.abs(E[:, 0]).max():.2e}')
print(f'max |H − H0| / |H0| (todos los casos) = {(np.abs(H - H[:, :1]).max(axis=1)/np.abs(H[:, 0])).max():.2e}   (H = E se conserva)')
ig = i_gen
print(f'caso general:  p_φ1 varía {np.ptp(pph1[ig]):.1e} ,  p_φ2 varía {np.ptp(pph2[ig]):.1e}   (se conservan)')
print(f'               p_r varía {np.ptp(pr[ig]):.3f} ,  p_θ varía {np.ptp(pth[ig]):.3f}   (no se conservan)')
for nombre, k_ in (('caso general', i_gen), ('animación (movimiento cualquiera)', i_anim_gen), ('animación (oscilaciones)', i_anim_osc)):
    d_max = (np.pi - th1[k_]).max()
    print(f'{nombre}: m1 dentro del cono: max(π − θ1) = {d_max:.3f} rad  <  α = {alpha:.3f} rad  ->  {d_max < alpha} ;'
          f'  r1 ∈ [{r1[k_].min():.3f}, {r1[k_].max():.3f}] m  ⊂ (0, ℓ = {ell})')

s = max(1, N // 20000)                                                    # submuestreo
fig, ax = plt.subplots(3, 1, figsize=(9, 9), sharex=True)

ax[0].plot(t[::s], E[ig, ::s], color='tab:blue', lw=3, label='E = T + U')
ax[0].plot(t[::s], H[ig, ::s], 'r--', lw=1.5, label='H(q, p) = Σ p q̇ − L')
ax[0].plot(t[::s], L[ig, ::s], color='tab:purple', lw=1.2, label='L = T − U  (no se conserva)')
ax[0].set(ylabel='J', title='Caso general: E, H y L'); ax[0].legend(fontsize=8)

for P, nm, col in ((pph1, 'p_φ1', 'tab:green'), (pph2, 'p_φ2', 'tab:orange'), (pr, 'p_r', 'tab:red'), (pth, 'p_θ', 'tab:gray')):
    ax[1].plot(t[::s], P[ig, ::s], color=col, lw=2, label=nm + ('  (cte)' if nm.startswith('p_φ') else ''))
ax[1].set(ylabel='momentos', title='Momentos generalizados'); ax[1].legend(fontsize=8, ncol=4)

eps = 1e-18                                                               # evita log(0)
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - H[ig, 0])/abs(H[ig, 0]) + eps, color='tab:red',   lw=1, label='|H − H₀| / |H₀|  (integrador RK4)')
ax[2].semilogy(t[::s], np.abs(H[ig, ::s] - E[ig, ::s])/abs(E[ig, 0]) + eps, color='tab:green', lw=1, label='|H − E| / |E₀|  (redondeo)')
ax[2].set(xlabel='t (s)', ylabel='error relativo', title='Error numérico'); ax[2].legend(fontsize=8, loc='lower right')

for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## (e) m2 EN TRAYECTORIA CIRCULAR: PEQUEÑAS OSCILACIONES DE m1

In [ ]:
#(e) m2 en trayectoria horizontal circular (r2 = cte  ->  r1 = r10)  y  φ̇1(0) = 0  (p_φ1 = 0  ->  φ1 = cte: m1 oscila en un plano)
#     radial:  (ℓ − r10) φ̇2² sin²α = g (1 − cosα)   ->   φ̇2 = sqrt( g / ((ℓ − r10)(1 + cosα)) )
#     θ1 = π − δ :   r10 δ̈ + g δ = 0                ->   δ(t) = δ0 cos(ω1 t),   ω1 = sqrt(g / r10)
phd2_an = np.sqrt(g/((ell - r10)*(1 + np.cos(alpha))))
print(f'ω1 = sqrt(g/r10) = {om1:.4f} rad/s   (T1 = {2*np.pi/om1:.4f} s)')
print(f'φ̇2 = sqrt(g/((ℓ−r10)(1+cosα))) = {phd2_an:.4f} rad/s   (T2 = {2*np.pi/phd2_an:.4f} s)')

delta   = np.pi - th1                                                      # δ(t) = π − θ1
deltas0 = np.pi - np.array([c_['th1'] for c_ in casos])
delta_an = deltas0[:4, None]*np.cos(om1*t)[None, :]
phi2_an  = phd2_an*t

print(f'\n{"δ0":>8}{"max|δ−δ_an|/δ0":>18}{"max|r1−r10|/δ0²":>18}{"max|φ2−φ2_an|":>16}{"max|φ1|":>10}')
for k in range(4):
    print(f'{deltas0[k]:>8.3f}{np.abs(delta[k] - delta_an[k]).max()/deltas0[k]:>18.2e}{np.abs(r1[k] - r10).max()/deltas0[k]**2:>18.3f}'
          f'{np.abs(ph2[k] - phi2_an).max():>16.2e}{np.abs(ph1[k]).max():>10.1e}')

s = max(1, N // 5000)
k = i_main
fig, ax = plt.subplots(2, 2, figsize=(11, 7))

ax[0,0].plot(t[::s], delta[k, ::s], lw=2, label='numérica')
ax[0,0].plot(t[::s], delta_an[k, ::s], 'r--', lw=1.2, label='δ0 cos(ω1 t)')
ax[0,0].set(xlabel='t (s)', ylabel='δ = π − θ1 (rad)', title='Oscilación de m1'); ax[0,0].legend(fontsize=8)

ax[0,1].plot(t[::s], (delta[k] - delta_an[k])[::s], color='tab:green')
ax[0,1].set(xlabel='t (s)', ylabel='δ_num − δ_an', title='Diferencia (desfase por no linealidad)')

ax[1,0].plot(t[::s], (r1[k] - r10)[::s], color='tab:purple')
ax[1,0].set(xlabel='t (s)', ylabel='r1 − r10 (m)', title=f'r1 ≈ cte  (variación ~ δ0² = {deltas0[k]**2:.3f})')

ax[1,1].plot(t[::s], ph2[k, ::s], lw=2, label='numérica')
ax[1,1].plot(t[::s], phi2_an[::s], 'r--', lw=1.2, label='φ̇2 t')
ax[1,1].set(xlabel='t (s)', ylabel='φ2 (rad)', title='Giro de m2'); ax[1,1].legend(fontsize=8)

for a in ax.ravel(): a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ANIMACIÓN (3D + dos planos 2D)

In [ ]:
from IPython.display import display
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto
_anims = []                                        # referencias a las animaciones (necesario en modo interactivo)

def animar(caso, interactivo=False, elev=20, azim=-60, rotar=0, T_anim=None, fps=15, speed=1.0, trail_s=2.0):
    """
    Anima el caso `caso` (índice en la lista `casos`): vista 3D + plano lateral (x-z) + plano cenital (x-y).
      interactivo : True  -> figura interactiva (requiere ipympl y %matplotlib widget): se gira la vista 3D con el ratón
                    False -> animación HTML (el ratón no gira la vista; usa `rotar`)
      elev, azim  : ángulos iniciales de la vista 3D (grados)
      rotar       : giro automático de la cámara (grados por segundo; 0 = fija)
      T_anim      : segundos simulados a animar (por defecto tf)
      fps, speed  : fotogramas por segundo y velocidad (1 = tiempo real, 0.5 = cámara lenta)
      trail_s     : duración de la estela de las masas (s); 0 = sin estela
    """
    T_anim = tf if T_anim is None else min(T_anim, tf)

    # Submuestreo: un fotograma cada 'step' pasos de simulación
    step    = max(1, round(speed / (fps * dt)))
    idx     = np.arange(0, min(N, int(T_anim / dt)), step)
    trail_n = int(trail_s / (dt * step))

    # Posiciones (z hacia arriba, O en el origen)
    ts = t[idx]
    r1a, th1a, ph1a, ph2a = r1[caso, idx], th1[caso, idx], ph1[caso, idx], ph2[caso, idx]
    r2a = ell - r1a
    x1, y1, z1 = r1a*np.sin(th1a)*np.cos(ph1a), r1a*np.sin(th1a)*np.sin(ph1a), r1a*np.cos(th1a)             # m1 (cuelga dentro del cono)
    x2, y2, z2 = r2a*np.sin(alpha)*np.cos(ph2a), r2a*np.sin(alpha)*np.sin(ph2a), -r2a*np.cos(alpha)         # m2 (sobre el cono)

    # Límites y cono
    depth = 1.1*max(-z1.min(), -z2.min()) + 0.1                  # profundidad dibujada del cono
    s_max = depth/np.cos(alpha)                                  # longitud de la generatriz dibujada
    Rl    = 1.1*s_max*np.sin(alpha)
    z_lo, z_hi = -depth, 0.15*depth
    phi = np.linspace(0, 2*np.pi, 100)

    fig = plt.figure(figsize=(14, 4.8))
    ax3 = fig.add_subplot(1, 3, 1, projection='3d')
    axs = fig.add_subplot(1, 3, 2)                     # plano lateral x-z
    axt = fig.add_subplot(1, 3, 3)                     # plano cenital x-y
    fig.subplots_adjust(left=0.02, right=0.98, wspace=0.35, top=0.88)
    fig.suptitle(casos[caso]['label'], fontsize=11)

    # --- Vista 3D: cono ---
    ax3.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), zlim=(z_lo, z_hi), xlabel='x', ylabel='y', zlabel='z', title='Vista 3D')
    ax3.set_box_aspect((2*Rl, 2*Rl, z_hi - z_lo))
    ax3.grid(False)                                    # sin rejilla
    ax3.view_init(elev=elev, azim=azim)

    Sc, Pc = np.meshgrid(np.linspace(0, s_max, 2), np.linspace(0, 2*np.pi, 80))
    ax3.plot_surface(Sc*np.sin(alpha)*np.cos(Pc), Sc*np.sin(alpha)*np.sin(Pc), -Sc*np.cos(alpha),
                     color='tab:blue', alpha=0.12, linewidth=0, shade=False)                                   # superficie del cono
    ax3.plot(s_max*np.sin(alpha)*np.cos(phi), s_max*np.sin(alpha)*np.sin(phi), -s_max*np.cos(alpha)*np.ones_like(phi), 'k-', lw=1.2)   # borde
    for a_ in np.linspace(0, 2*np.pi, 13)[:-1]:
        ax3.plot([0, s_max*np.sin(alpha)*np.cos(a_)], [0, s_max*np.sin(alpha)*np.sin(a_)], [0, -s_max*np.cos(alpha)],
                 color='tab:blue', lw=0.6, alpha=0.6)                                                         # generatrices
    ax3.plot([0, 0], [0, 0], [z_lo, z_hi], 'k:', lw=1)  # eje z
    ax3.plot([0], [0], [0], 'ko', ms=4)                 # vértice O

    # --- Plano lateral: generatrices del cono ---
    axs.set(xlim=(-Rl, Rl), ylim=(z_lo, z_hi), aspect='equal', xlabel='x (m)', ylabel='z (m)', title='Plano lateral (x-z)')
    axs.plot([-s_max*np.sin(alpha), 0, s_max*np.sin(alpha)], [-s_max*np.cos(alpha), 0, -s_max*np.cos(alpha)], 'k-', lw=1.5)
    axs.fill([-s_max*np.sin(alpha), 0, s_max*np.sin(alpha)], [-s_max*np.cos(alpha), 0, -s_max*np.cos(alpha)], color='tab:blue', alpha=0.08)
    axs.axvline(0, color='k', ls=':', lw=1); axs.plot(0, 0, 'ko', ms=4); axs.grid(alpha=0.3)

    # --- Plano cenital: borde del cono ---
    axt.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Plano cenital (x-y)')
    axt.plot(s_max*np.sin(alpha)*np.cos(phi), s_max*np.sin(alpha)*np.sin(phi), 'k-', lw=1.2)
    axt.plot(0, 0, 'ko', ms=4); axt.grid(alpha=0.3)

    # Artistas
    str3,  = ax3.plot([], [], [], '-', color='dimgray', lw=1.5)                 # cuerda m1 - O - m2
    m1_3,  = ax3.plot([], [], [], 'o', color='tab:blue', mec='black', ms=9)
    m2_3,  = ax3.plot([], [], [], 'o', color='tab:red', mec='black', ms=9)
    tr1_3, = ax3.plot([], [], [], '-', color='tab:blue', lw=1, alpha=0.6)
    tr2_3, = ax3.plot([], [], [], '-', color='tab:red', lw=1, alpha=0.6)

    strS,  = axs.plot([], [], '-', color='dimgray', lw=1.5)
    m1_S,  = axs.plot([], [], 'o', color='tab:blue', mec='black', ms=9)
    m2_S,  = axs.plot([], [], 'o', color='tab:red', mec='black', ms=9)
    tr1_S, = axs.plot([], [], '-', color='tab:blue', lw=1, alpha=0.6)
    tr2_S, = axs.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)

    strT,  = axt.plot([], [], '-', color='dimgray', lw=1.5)
    m1_T,  = axt.plot([], [], 'o', color='tab:blue', mec='black', ms=9)
    m2_T,  = axt.plot([], [], 'o', color='tab:red', mec='black', ms=9)
    tr1_T, = axt.plot([], [], '-', color='tab:blue', lw=1, alpha=0.6)
    tr2_T, = axt.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)

    time_txt = fig.text(0.02, 0.93, '')

    def update(k):
        j = max(0, k - trail_n)
        str3.set_data_3d([x1[k], 0, x2[k]], [y1[k], 0, y2[k]], [z1[k], 0, z2[k]])
        m1_3.set_data_3d([x1[k]], [y1[k]], [z1[k]]);   m2_3.set_data_3d([x2[k]], [y2[k]], [z2[k]])
        tr1_3.set_data_3d(x1[j:k+1], y1[j:k+1], z1[j:k+1]);  tr2_3.set_data_3d(x2[j:k+1], y2[j:k+1], z2[j:k+1])

        strS.set_data([x1[k], 0, x2[k]], [z1[k], 0, z2[k]])
        m1_S.set_data([x1[k]], [z1[k]]);  m2_S.set_data([x2[k]], [z2[k]])
        tr1_S.set_data(x1[j:k+1], z1[j:k+1]);  tr2_S.set_data(x2[j:k+1], z2[j:k+1])

        strT.set_data([x1[k], 0, x2[k]], [y1[k], 0, y2[k]])
        m1_T.set_data([x1[k]], [y1[k]]);  m2_T.set_data([x2[k]], [y2[k]])
        tr1_T.set_data(x1[j:k+1], y1[j:k+1]);  tr2_T.set_data(x2[j:k+1], y2[j:k+1])

        if rotar != 0:                                 # solo si se pide giro automático (si no, se respeta la vista del ratón)
            ax3.view_init(elev=elev, azim=azim + rotar*ts[k])

        time_txt.set_text(f't = {ts[k]:.2f} s    r1 = {r1a[k]:.3f} m    θ1 = {np.degrees(th1a[k]):.1f}°    φ2 = {np.degrees(ph2a[k]) % 360:.0f}°')
        return str3, m1_3, m2_3, tr1_3, tr2_3, strS, m1_S, m2_S, tr1_S, tr2_S, strT, m1_T, m2_T, tr1_T, tr2_T, time_txt

    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=False)   # blit=False: necesario con ejes 3D
    _anims.append(anim)

    if interactivo:
        plt.show()                                     # no cerrar la figura: es la que se maneja con el ratón
    else:
        plt.close(fig)
        display(HTML(anim.to_jshtml()))
    return anim

### Animación 1: pequeñas oscilaciones de m1 (m2 en círculo horizontal)

In [ ]:
interactivo = False    # True: figura interactiva (gira la vista 3D con el ratón; requiere ipympl) / False: animación HTML
if interactivo:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')      # backend interactivo:  pip install ipympl
    except Exception as e:
        interactivo = False
        print('Recordatorio para el modo interactivo (instalar ipympl).', e)

animar(i_anim_osc, interactivo=interactivo, elev=20, azim=-60, rotar=0, fps=15, speed=1.0, trail_s=2.0);

### Animación 2: movimiento cualquiera (m1 en 3D)

In [ ]:
animar(i_anim_gen, interactivo=interactivo, elev=20, azim=-60, rotar=0, fps=15, speed=1.0, trail_s=2.0);